In [ ]:
# # =========================================
# # Imports
# # =========================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import evaluate

# =========================================
# Settings
# =========================================
MODEL_NAME = "t5-small"  # small and memory-friendly
MAX_INPUT_LEN = 64  # 128
MAX_TARGET_LEN = 32  # 64
BATCH_SIZE = 64  # chat wanted me to put 1 ??? thats so dumb and inefficient
GRAD_ACCUM = 1  # 8    this means that the gradients are not all calculated each time
EPOCHS = 10
LEARNING_RATE = 3e-4

# =========================================
# Load tokenizer & model
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# =========================================
# Load your dataset
# Assume TSV: english \t toki_pona
# =========================================
data_files = {
    "train": "../data/train.tsv",
    "validation": "../data/val.tsv",
    "test": "../data/test.tsv"
}

dataset = load_dataset("csv", data_files=data_files, delimiter="\t",
                       column_names=["english", "toki_pona"])

# =========================================
# Normalization (optional but safe)
# =========================================
# def normalize_tp(text):
#     text = text.lower()
#     text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

# =========================================
# Preprocessing / Tokenization
# =========================================
def preprocess(batch):
    inputs = [f"translate English to Toki Pona: {en}" for en in batch["english"]]
    #targets = [normalize_tp(tp) for tp in batch["toki_pona"]]  # normalisation was not necessary
    targets = [tp for tp in batch["toki_pona"]]

    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(targets, max_length=MAX_TARGET_LEN, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

# =========================================
# Data collator
# =========================================
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# =========================================
# Training Arguments
# =========================================
training_args = TrainingArguments(
    output_dir="./models/t5-honyak",  # model saved here every epoch
    eval_strategy="epoch",
    save_strategy="epoch",  # saves every epoch
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    log_level="info",
    save_total_limit=2,
    fp16=True,  # works on M1, reduces memory
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

# =========================================
# Trainer
# =========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

loading file spiece.model from cache at /Users/twuj/.cache/huggingface/hub/models--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4/spiece.model
loading file tokenizer.json from cache at /Users/twuj/.cache/huggingface/hub/models--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /Users/twuj/.cache/huggingface/hub/models--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4/tokenizer_config.json
loading file chat_template.jinja from cache at None
loading configuration file config.json from cache at /Users/twuj/.cache/huggingface/hub/models--t5-small/snapshots/df1b051c49625cf57a3d0d8d3863ed4d13564fe4/config.json
Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_i

In [ ]:
# Train t5-honyak-10-epoch 
trainer.train(resume_from_checkpoint=True)

# last 7 epohcs took 1h to train (resumed from checkpoint)

Loading model from ./t5-honyak/checkpoint-1059.
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
***** Running training *****
  Num examples = 22,592
  Num Epochs = 10
  Instantaneous batch size per device = 64
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 1
  Total optimization steps = 3,530
  Number of trainable parameters = 60,506,624
  Continuing training from checkpoint, will skip to saved global_step
  Continuing training from epoch 3
  Continuing training from global step 1059
  Will skip the first 3 epochs then the first 0 batches in the first epoch.
/opt/homebrew/anaconda3/envs/honyak/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
4,0.833200,0.713268
5,0.761600,0.649397
6,0.698100,0.621905
7,0.666300,0.591680
8,0.638100,0.572081
9,0.613400,0.567713
10,0.584100,0.567872



***** Running Evaluation *****
  Num examples = 2824
  Batch size = 64
Saving model checkpoint to ./t5-honyak/checkpoint-1412
Configuration saved in ./t5-honyak/checkpoint-1412/config.json
Configuration saved in ./t5-honyak/checkpoint-1412/generation_config.json
Model weights saved in ./t5-honyak/checkpoint-1412/model.safetensors
tokenizer config file saved in ./t5-honyak/checkpoint-1412/tokenizer_config.json
Special tokens file saved in ./t5-honyak/checkpoint-1412/special_tokens_map.json
Copy vocab file to ./t5-honyak/checkpoint-1412/spiece.model
Deleting older checkpoint [t5-honyak/checkpoint-706] due to args.save_total_limit
/opt/homebrew/anaconda3/envs/honyak/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)

***** Running Evaluation *****
  Num examples = 2824
  Batch size = 64
Saving model checkpoint to ./t5-honyak/checkpoin

TrainOutput(global_step=3530, training_loss=0.49112706116846533, metrics={'train_runtime': 3981.6968, 'train_samples_per_second': 56.74, 'train_steps_per_second': 0.887, 'total_flos': 2111890387894272.0, 'train_loss': 0.49112706116846533, 'epoch': 10.0})

In [ ]:
# with 1% dataset, one epoch, took 10 secs
# with 10% dataset, one epoch, took 3 mins with validation ?
# 10% dataset, 3 epochs, 9mins
# 1% dataset, 10 epochs, 2 mins

# =========================================
# Save model & tokenizer
# ==================================cod=======
model_name = "models/t5-honyak-example"
trainer.save_model(model_name)
tokenizer.save_pretrained(model_name)

Saving model checkpoint to t5-honyak-10-epoch
Configuration saved in t5-honyak-10-epoch/config.json
Configuration saved in t5-honyak-10-epoch/generation_config.json
Model weights saved in t5-honyak-10-epoch/model.safetensors
tokenizer config file saved in t5-honyak-10-epoch/tokenizer_config.json
Special tokens file saved in t5-honyak-10-epoch/special_tokens_map.json
Copy vocab file to t5-honyak-10-epoch/spiece.model
tokenizer config file saved in t5-honyak-10-epoch/tokenizer_config.json
Special tokens file saved in t5-honyak-10-epoch/special_tokens_map.json
Copy vocab file to t5-honyak-10-epoch/spiece.model


('t5-honyak-10-epoch/tokenizer_config.json',
 't5-honyak-10-epoch/special_tokens_map.json',
 't5-honyak-10-epoch/spiece.model',
 't5-honyak-10-epoch/added_tokens.json',
 't5-honyak-10-epoch/tokenizer.json')